In [ ]:
# WARNING: Sample 'WT_IEL_CD69lo_CD103lo_1_S19_L005_R1_001.fastq.gz' is incomplete
# We performed manual Trim Galore! before starting NextFlow pipeline

# specify the directory containing the input files
INPUT_DIR=~/projects/190703_ATAC_SQ_TGFbrKO/

# specify the output directory
OUTPUT_DIR=./TrimGalore/

# loop through all .fastq.gz files in the input directory
for file in $INPUT_DIR/WT_*.fastq.gz
do
  # trim the adapter sequences from the input file
  trim_galore $file -o $OUTPUT_DIR --phred33
done

In [ ]:
# WARNING: Sample 'WT_IEL_CD69lo_CD103lo_1_S19_L005_R1_001.fastq.gz' is incomplete
# Run Nextflow for ATAC-Seq skipping FastQC and trimming to avoid errors
# Useful reference: https://nf-co.re/atacseq
./nextflow run nf-core/atacseq --input samples.csv --genome mm10 -profile singularity --read_length 75 --outdir output --skip_fastqc --skip_trimming --email ggalletti@ucsd.edu -resume

- Running the DESeq2 pipeline on the FeatureCounts file showed low degree of similarity between replicates 
- Useful reference: [Introduction to DESeq2](introduction-to-deseq2/index.qmd)

-- The following analysis is performed to improve the similarity of replicates --

In [ ]:
# To run IDR the narrowPeak files have to be sorted by the -log10(p-value) column
# Useful reference: https://hbctraining.github.io/Intro-to-ChIPseq/lessons/07_handling-replicates-idr.html
sort -k8,8nr IEL_69h103h_REP1.mLb.clN_peaks.broadPeak > ~/projects/190703_ATAC_SQ_TGFbrKO/idr/IEL_69h103h_REP1.mLb.clN_peaks.broadPeak
sort -k8,8nr IEL_69h103h_REP2.mLb.clN_peaks.broadPeak > ~/projects/190703_ATAC_SQ_TGFbrKO/idr/IEL_69h103h_REP2.mLb.clN_peaks.broadPeak
sort -k8,8nr IEL_69h103l_REP1.mLb.clN_peaks.broadPeak > ~/projects/190703_ATAC_SQ_TGFbrKO/idr/IEL_69h103l_REP1.mLb.clN_peaks.broadPeak
sort -k8,8nr IEL_69h103l_REP2.mLb.clN_peaks.broadPeak > ~/projects/190703_ATAC_SQ_TGFbrKO/idr/IEL_69h103l_REP2.mLb.clN_peaks.broadPeak
sort -k8,8nr IEL_69l103l_REP1.mLb.clN_peaks.broadPeak > ~/projects/190703_ATAC_SQ_TGFbrKO/idr/IEL_69l103l_REP1.mLb.clN_peaks.broadPeak
sort -k8,8nr IEL_69l103l_REP2.mLb.clN_peaks.broadPeak > ~/projects/190703_ATAC_SQ_TGFbrKO/idr/IEL_69l103l_REP2.mLb.clN_peaks.broadPeak
sort -k8,8nr SP_MPEC_REP1.mLb.clN_peaks.broadPeak > ~/projects/190703_ATAC_SQ_TGFbrKO/idr/SP_MPEC_REP1.mLb.clN_peaks.broadPeak
sort -k8,8nr SP_MPEC_REP2.mLb.clN_peaks.broadPeak > ~/projects/190703_ATAC_SQ_TGFbrKO/idr/SP_MPEC_REP2.mLb.clN_peaks.broadPeak

#Run IDR on the replicates
module load gcc python3essential idr

idr --samples ./raw_sorted/IEL_69h103h_REP1.mLb.clN_peaks.broadPeak ./raw_sorted/IEL_69h103h_REP2.mLb.clN_peaks.broadPeak \
--input-file-type broadPeak \
--rank p.value \
--output-file 69h103h-idr \
--plot \
--log-output-file 69h103h.idr.log

idr --samples ./raw_sorted/IEL_69h103l_REP1.mLb.clN_peaks.broadPeak ./raw_sorted/IEL_69h103l_REP2.mLb.clN_peaks.broadPeak \
--input-file-type broadPeak \
--rank p.value \
--output-file 69h103l-idr \
--plot \
--log-output-file 69h103l.idr.log

idr --samples ./raw_sorted/IEL_69l103l_REP1.mLb.clN_peaks.broadPeak ./raw_sorted/IEL_69l103l_REP2.mLb.clN_peaks.broadPeak \
--input-file-type broadPeak \
--rank p.value \
--output-file 69l103l-idr \
--plot \
--log-output-file 69l103l.idr.log

idr --samples ./raw_sorted/SP_MPEC_REP1.mLb.clN_peaks.broadPeak ./raw_sorted/SP_MPEC_REP2.mLb.clN_peaks.broadPeak \
--input-file-type broadPeak \
--rank p.value \
--output-file MPEC-idr \
--plot \
--log-output-file MPEC.idr.log

In [ ]:
# Run DiffBind to generate new count file with improved peaks
# Useful reference: https://hbctraining.github.io/In-depth-NGS-Data-Analysis-Course/sessionV/lessons/08_diffbind_differential_peaks.html

library("DiffBind")
library("tidyverse")
library("ChIPpeakAnno")
library("ChIPseeker")
library("GenomicRanges")
library("org.Mm.eg.db")
library("TxDb.Mmusculus.UCSC.mm10.knownGene")
library("biomaRt")

samples <- read.csv('annotation.csv')
dbObj <- dba(sampleSheet=samples)
dbObj

dbObj <- dba.count(dbObj, bUseSummarizeOverlaps=TRUE)

pdf("dbaPCA.pdf", 12, 7)
dba.plotPCA(dbObj,  attributes=DBA_FACTOR, label=DBA_ID)
plot(dbObj)
dev.off()

counts <- dba.peakset(dbObj, bRetrieve=T, DataType=DBA_DATA_FRAME)
write.table(counts, file="./featureCounts.txt", sep="\t", quote=F, row.names=F)

In [ ]:
# Generate an annotation file with the new intervals annotated

# Convert to GRanges
gr <- makeGRangesFromDataFrame(counts, ignore.strand = T, seqnames.field = "CHR",
                                 start.field = "START", end.field = "END")

# Give ranges numeric names in order
names(gr) <- c(1:length(gr))

# Create GRanges object with annotations from TxDb database
annoData <- toGRanges(TxDb.Mmusculus.UCSC.mm10.knownGene, feature="gene")

# Annotate granges with the nearest TSS
annot <- annotatePeakInBatch(gr, 
                               AnnotationData=annoData, 
                               featureType = "TSS",
                               output="nearestLocation",
                               PeakLocForDistance = "start")

# annotate with ChIPseeker
peak.anno <- annotatePeak(peak = annot, tssRegion = c(-1000, 1000), TxDb = TxDb.Mmusculus.UCSC.mm10.knownGene, annoDb = "org.Mm.eg.db")

pdf("peak_anno.pdf", 12, 7)
plotAnnoPie(peak.anno)
dev.off()

dfPA = as.data.frame(peak.anno)
write.table(dfPA, file="./peak_anno.csv", col.names = T)

In [ ]:
# Perform the standard DESeq2 pipeline on the new FeatureCounts file
# Useful reference: https://protocols.beetletun.de/docs/bioinformatics/introduction-to-deseq2/

# BiocManager is an excellent source for R packages used in bioinformatics
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

# these are the packages that we are going to need.
list.of.packages <- c("ggplot2", "DESeq2", "readr", "readxl", "pheatmap",
                      "RColorBrewer", "EnhancedVolcano", "fgsea", "org.Mm.eg.db",
                      "AnnotationDbi", "gage", "plotly", "dplyr", "ComplexHeatmap",
                      "patchwork", "GSVA", "limma", "magrittr", "purrr")

# check, which packages have not been installed so far
new.packages <- list.of.packages[!(list.of.packages %in% installed.packages()[,"Package"])]
# install the missing packages. If a package is not in Bioconductor, it will be installed from cran. this can take a while
if(length(new.packages)) BiocManager::install(new.packages)

In [ ]:
# Load libraries
.libPaths( c( "~/R/x86_64-pc-linux-gnu-library/3.6" , .libPaths() ) )
library(readr)
library(dplyr)

# Load the FeatureCount file
counts <- read.table("./featureCounts.txt", header = T)
colnames(counts) <- c('Chr','Start','End','69H_103H_1','69H_103H_2','69H_103L_1','69H_103L_2','69L_103L_1','69L_103L_2','MPEC_1','MPEC_2')

# Annotate the FeatureCount file
annot <- read.table("./peak_anno.csv")
annot <- select(annot, seqnames, start, end, annotation, distanceToTSS, SYMBOL, ENSEMBL)
colnames(annot) <- c('Chr','Start','End','Annot','DistToTSS','GeneSymbol','ENS_ID')
counts_temp <- counts %>% inner_join(annot, by=c('Chr','Start','End'))
counts_temp <- counts_temp[!is.na(counts_temp$GeneSymbol),]
counts_temp$Interval <- rownames(counts_temp)
counts_temp[rownames(counts_temp) == '12455',]
annot[rownames(annot) == '12455',]
counts <- counts_temp

# Load the sample annotation file
annotation <- read_csv("/home/ggalletti/projects/190703_ATAC_SQ_TGFbrKO/TrimGalore/output/bwa/merged_replicate/macs2/broad_peak/consensus/annotation.csv")

head(counts)
head(annotation)

# to set to rownames, we first have to convert the table to a data.frame
counts <- as.data.frame(counts)

# Set the column Geneid as the rownames of the table.
rownames(counts) <- counts$Interval

# Remove GeneSymbol from the table
counts$Interval <- NULL

head(counts)

# set rownames
annotation <- as.data.frame(annotation)
rownames(annotation) <- annotation$sample
annotation$sample <- NULL

# check if all rownames in annotation correspond to a column in counts
all(rownames(annotation) %in% colnames(counts))  # --> this should return TRUE

# check if they are in the same order
# if this does not return TRUE, do not worry about that. We will sort the columns of counts accordingly.
all(rownames(annotation) == colnames(counts))

# reorder the columns of counts
counts <- counts[, rownames(annotation)]

# now, this sould return TRUE
all(rownames(annotation) == colnames(counts))

In [ ]:
suppressPackageStartupMessages(library(DESeq2))

dds <- DESeqDataSetFromMatrix(countData = round(counts),
                              colData = annotation,
                              design = ~ subset)

# Annotate the deseq2 dataset
match_rows <- match(row.names(annot), row.names(dds))
annot <- annot[order(match_rows),]
metadata <- DataFrame(GeneSymbol=annot$GeneSymbol, Annotation=annot$Annot, DistToTSS=annot$DistToTSS, ENS_ID=annot$ENS_ID)
mcols(dds) <- metadata

dds <- DESeq(dds)

dds

keep <- rowSums(counts(dds)) >= 10
dds <- dds[keep,]

In [ ]:
# PCA

ntd <- normTransform(dds)  ## log2(n +1)
vsd <- vst(dds, blind=FALSE)
rld <- rlog(dds, blind=FALSE)

pdf("PCA.pdf", 12, 7)
plotPCA(ntd, intgroup=c("subset")) + ggplot2::theme_bw()
plotPCA(vsd, intgroup=c("subset")) + ggplot2::theme_bw()
plotPCA(rld, intgroup=c("subset")) + ggplot2::theme_bw()
dev.off()

In [ ]:
# Sample distance matrix

library(pheatmap)
library(RColorBrewer)

sampleDists <- dist(t(assay(vsd)))
sampleDistMatrix <- as.matrix(sampleDists)
rownames(sampleDistMatrix) <- paste(vsd$sample, vsd$tissue, vsd$site, sep="-")
colnames(sampleDistMatrix) <- NULL
colors <- colorRampPalette( rev(brewer.pal(9, "Blues")) )(255)

pdf("Distance.pdf", 12, 7)
pheatmap(sampleDistMatrix,
         clustering_distance_rows=sampleDists,
         clustering_distance_cols=sampleDists,
         col=colors)
dev.off()

In [ ]:
# create new grouping variable and convert it to a factor
dds$subset_tissue <- factor(paste(dds$subset, dds$tissue, sep="-"))
dds$subset_tissue

# update design
design(dds) <- ~ subset_tissue
design(dds)

dds <- DESeq(dds)

# we want to compare all the conditions
res_HH_HL <- results(dds, contrast = c("subset_tissue", "69H_103H-IEL", "69H_103L-IEL"))
res_HH_LL <- results(dds, contrast = c("subset_tissue", "69H_103H-IEL", "69L_103L-IEL"))
res_HL_LL <- results(dds, contrast = c("subset_tissue", "69H_103L-IEL", "69L_103L-IEL"))
res_HH_MPEC <- results(dds, contrast = c("subset_tissue", "69H_103H-IEL", "MPEC-SPL"))
res_HL_MPEC <- results(dds, contrast = c("subset_tissue", "69H_103L-IEL", "MPEC-SPL"))
res_LL_MPEC <- results(dds, contrast = c("subset_tissue", "69L_103L-IEL", "MPEC-SPL"))

pdf("MA.pdf", 12, 7)
plotMA(res_HH_HL, ylim=c(-2,2))
plotMA(res_HH_LL, ylim=c(-2,2))
plotMA(res_HL_LL, ylim=c(-2,2))
plotMA(res_HH_MPEC, ylim=c(-2,2))
plotMA(res_HL_MPEC, ylim=c(-2,2))
plotMA(res_LL_MPEC, ylim=c(-2,2))
dev.off()

resLFC_HH_HL <- lfcShrink(dds, contrast = c("subset_tissue", "69H_103H-IEL", "69H_103L-IEL"), type="normal")
resLFC_HH_LL <- lfcShrink(dds, contrast = c("subset_tissue", "69H_103H-IEL", "69L_103L-IEL"), type="normal")
resLFC_HL_LL <- lfcShrink(dds, contrast = c("subset_tissue", "69H_103L-IEL", "69L_103L-IEL"), type="normal")
resLFC_HH_MPEC <- lfcShrink(dds, contrast = c("subset_tissue", "69H_103H-IEL", "MPEC-SPL"))
resLFC_HL_MPEC <- lfcShrink(dds, contrast = c("subset_tissue", "69H_103L-IEL", "MPEC-SPL"))
resLFC_LL_MPEC <- lfcShrink(dds, contrast = c("subset_tissue", "69L_103L-IEL", "MPEC-SPL"))
resLFC_HH_HL
resLFC_HH_LL
resLFC_HL_LL
resLFC_HH_MPEC
resLFC_HL_MPEC
resLFC_LL_MPEC

library(EnhancedVolcano)

# Generate a report pdf with QC plots and Volcano plots
pdf("LFC_plots.pdf", 6, 8)
plotMA(resLFC_HH_HL, ylim=c(-2,2))
plotMA(resLFC_HH_LL, ylim=c(-2,2))
plotMA(resLFC_HL_LL, ylim=c(-2,2))
plotMA(resLFC_HH_MPEC, ylim=c(-2,2))
plotMA(resLFC_HL_MPEC, ylim=c(-2,2))
plotMA(resLFC_LL_MPEC, ylim=c(-2,2))
plot(res_HH_HL$pvalue, resLFC_HH_HL$pvalue)
plot(res_HH_LL$pvalue, resLFC_HH_LL$pvalue)
plot(res_HL_LL$pvalue, resLFC_HL_LL$pvalue)
plot(res_HH_MPEC$pvalue, resLFC_HH_MPEC$pvalue)
plot(res_HL_MPEC$pvalue, resLFC_HL_MPEC$pvalue)
plot(res_LL_MPEC$pvalue, resLFC_LL_MPEC$pvalue)
EnhancedVolcano(resLFC_HH_HL,
    lab = mcols(dds)$GeneSymbol,
    x = 'log2FoldChange',
    y = 'padj', ## use pvalue for unadjusted p values
    ylim = c(-0.5,6),
    pCutoff = 10e-2,
    title = "Volcano plot",
    subtitle = "Comparison HH vs HL")
EnhancedVolcano(resLFC_HH_LL,
    lab = mcols(dds)$GeneSymbol,
    x = 'log2FoldChange',
    y = 'padj', ## use pvalue for unadjusted p values
    ylim = c(-0.5,6),
    pCutoff = 10e-2,
    title = "Volcano plot",
    subtitle = "Comparison HH vs LL")
EnhancedVolcano(resLFC_HL_LL,
    lab = mcols(dds)$GeneSymbol,
    x = 'log2FoldChange',
    y = 'padj', ## use pvalue for unadjusted p values
    ylim = c(-0.5,4),
    pCutoff = 10e-2,
    title = "Volcano plot",
    subtitle = "Comparison HL vs LL")
EnhancedVolcano(resLFC_HH_MPEC,
    lab = mcols(dds)$GeneSymbol,
    x = 'log2FoldChange',
    y = 'padj', ## use pvalue for unadjusted p values
    ylim = c(-0.5,40),
    pCutoff = 10e-2,
    title = "Volcano plot",
    subtitle = "Comparison HH vs MPEC")
EnhancedVolcano(resLFC_HL_MPEC,
    lab = mcols(dds)$GeneSymbol,
    x = 'log2FoldChange',
    y = 'padj', ## use pvalue for unadjusted p values
    #ylim = c(-1,5),
    pCutoff = 10e-2,
    title = "Volcano plot",
    subtitle = "Comparison HL vs MPEC")
EnhancedVolcano(resLFC_LL_MPEC,
    lab = mcols(dds)$GeneSymbol,
    x = 'log2FoldChange',
    y = 'padj', ## use pvalue for unadjusted p values
    ylim = c(-0.5,12.5),
    pCutoff = 10e-2,
    title = "Volcano plot",
    subtitle = "Comparison LL vs MPEC")
plotCounts(dds, gene=which.min(resLFC_HH_HL$padj), intgroup="subset_tissue")
plotCounts(dds, gene=which.min(resLFC_HH_LL$padj), intgroup="subset_tissue")
plotCounts(dds, gene=which.min(resLFC_HL_LL$padj), intgroup="subset_tissue")
plotCounts(dds, gene=which.min(resLFC_HH_MPEC$padj), intgroup="subset_tissue")
plotCounts(dds, gene=which.min(resLFC_HL_MPEC$padj), intgroup="subset_tissue")
plotCounts(dds, gene=which.min(resLFC_LL_MPEC$padj), intgroup="subset_tissue")
dev.off()

summary(resLFC_HH_HL)
summary(resLFC_HH_LL)
summary(resLFC_HL_LL)
summary(resLFC_HH_MPEC)
summary(resLFC_HL_MPEC)
summary(resLFC_LL_MPEC)

In [ ]:
## important: check columns before you change the order of the results!
resLFC_HH_HL$Intervals <- rownames(resLFC_HH_HL)
resLFC_HH_HL$GeneSymbol <- mcols(dds)$GeneSymbol
resLFC_HH_HL$Annotation <- mcols(dds)$Annotation
resLFC_HH_HL$DistToTSS <- mcols(dds)$DistToTSS
resLFC_HH_HL$ENS_ID <- mcols(dds)$ENS_ID
head(resLFC_HH_HL)
resOrdered_HH_HL <- resLFC_HH_HL[order(resLFC_HH_HL$padj),]
resOrdered_HH_HL <- na.omit(resOrdered_HH_HL)
head(resOrdered_HH_HL)
write_csv(as.data.frame(resOrdered_HH_HL), "de_HH_HL.csv")

resLFC_HH_LL$Intervals <- rownames(resLFC_HH_LL)
resLFC_HH_LL$GeneSymbol <- mcols(dds)$GeneSymbol
resLFC_HH_LL$Annotation <- mcols(dds)$Annotation
resLFC_HH_LL$DistToTSS <- mcols(dds)$DistToTSS
resLFC_HH_LL$ENS_ID <- mcols(dds)$ENS_ID
head(resLFC_HH_LL)
resOrdered_HH_LL <- resLFC_HH_LL[order(resLFC_HH_LL$padj),]
resOrdered_HH_LL <- na.omit(resOrdered_HH_LL)
head(resOrdered_HH_LL)
write_csv(as.data.frame(resOrdered_HH_LL), "de_HH_LL.csv")

resLFC_HL_LL$Intervals <- rownames(resLFC_HL_LL)
resLFC_HL_LL$GeneSymbol <- mcols(dds)$GeneSymbol
resLFC_HL_LL$Annotation <- mcols(dds)$Annotation
resLFC_HL_LL$DistToTSS <- mcols(dds)$DistToTSS
resLFC_HL_LL$ENS_ID <- mcols(dds)$ENS_ID
head(resLFC_HL_LL)
resOrdered_HL_LL <- resLFC_HL_LL[order(resLFC_HL_LL$padj),]
resOrdered_HL_LL <- na.omit(resOrdered_HL_LL)
head(resOrdered_HL_LL)
write_csv(as.data.frame(resOrdered_HL_LL), "de_HL_LL.csv")

resLFC_HH_MPEC$Intervals <- rownames(resLFC_HH_MPEC)
resLFC_HH_MPEC$GeneSymbol <- mcols(dds)$GeneSymbol
resLFC_HH_MPEC$Annotation <- mcols(dds)$Annotation
resLFC_HH_MPEC$DistToTSS <- mcols(dds)$DistToTSS
resLFC_HH_MPEC$ENS_ID <- mcols(dds)$ENS_ID
head(resLFC_HH_MPEC)
resOrdered_HH_MPEC <- resLFC_HH_MPEC[order(resLFC_HH_MPEC$padj),]
resOrdered_HH_MPEC <- na.omit(resOrdered_HH_MPEC)
head(resOrdered_HH_MPEC)
write_csv(as.data.frame(resOrdered_HH_MPEC), "de_HH_MPEC.csv")

resLFC_HL_MPEC$Intervals <- rownames(resLFC_HL_MPEC)
resLFC_HL_MPEC$GeneSymbol <- mcols(dds)$GeneSymbol
resLFC_HL_MPEC$Annotation <- mcols(dds)$Annotation
resLFC_HL_MPEC$DistToTSS <- mcols(dds)$DistToTSS
resLFC_HL_MPEC$ENS_ID <- mcols(dds)$ENS_ID
head(resLFC_HL_MPEC)
resOrdered_HL_MPEC <- resLFC_HL_MPEC[order(resLFC_HL_MPEC$padj),]
resOrdered_HL_MPEC <- na.omit(resOrdered_HL_MPEC)
head(resOrdered_HL_MPEC)
write_csv(as.data.frame(resOrdered_HL_MPEC), "de_HL_MPEC.csv")

resLFC_LL_MPEC$Intervals <- rownames(resLFC_LL_MPEC)
resLFC_LL_MPEC$GeneSymbol <- mcols(dds)$GeneSymbol
resLFC_LL_MPEC$Annotation <- mcols(dds)$Annotation
resLFC_LL_MPEC$DistToTSS <- mcols(dds)$DistToTSS
resLFC_LL_MPEC$ENS_ID <- mcols(dds)$ENS_ID
head(resLFC_LL_MPEC)
resOrdered_LL_MPEC <- resLFC_LL_MPEC[order(resLFC_LL_MPEC$padj),]
resOrdered_LL_MPEC <- na.omit(resOrdered_LL_MPEC)
head(resOrdered_LL_MPEC)
write_csv(as.data.frame(resOrdered_LL_MPEC), "de_LL_MPEC.csv")

In [ ]:
# Generate a heatmap including all samples vs all samples and annotating the gene of interest
ddsLRT <- DESeq(dds, test="LRT", reduced= ~ 1)
resLRT <- results(ddsLRT)

# we are using which here, as some pvalues are NA.
de_intervals <- rownames(resLRT)[which(resLRT$padj<0.05)]

# extract count data and filter rows.
m <- assay(ntd)[de_intervals,]  ## or rld (for rlog) or vsd (for vst)

# keep only the samples, that we compared.
#m <- m[, dds$subset_tissue %in% c("69H_103H-IEL", "69H_103L-IEL", "69L_103L-IEL", "MPEC-SPL")]

suppressPackageStartupMessages(library(ComplexHeatmap))

# Identify the intervals/peaks corresponding to your gene(s) of interest
Rasa3 <- annot[annot$GeneSymbol == 'Rasa3', ]
Rasa3 <- Rasa3[!is.na(Rasa3$GeneSymbol),]
Rasa3 <- select(Rasa3, GeneSymbol, Intervals)
selected_intervals <- which(rownames(m) %in% c(rownames(Rasa3)))
selected_intervals

# Run heatmap
Heatmap <- Heatmap(t(scale(t(m))),     # we want to normalize the counts per row. 
                            # scale() normalizes per column.
                            # therefore we first transponse the matrix, 
                            # then call scale and then transpose again.
        col = c('white', 'light blue', 'blue'),
        show_row_names = F,
        #row_names_gp = gpar(fontsize = 1),
        #column_split = 2,
        #row_split = 2,
        name="z-score") 

# Plot heatmap and be sure to manually write down the name of the gene(s) as many times as the number of intervals you have
# Did not want to spend more time figuring out how to automatically markdown the gene name
pdf("heatmap.pdf", 6, 7)
Heatmap + rowAnnotation(link = row_anno_link(at = c(selected_intervals), labels = c("Rasa3","Rasa3","Rasa3","Rasa3","Rasa3","Rasa3","Rasa3")))
dev.off()